In [78]:
itest='Alaska_TXx_ij_glostLocSca_extremes_gev_CV_CI300'

In [79]:
import os

# Set OMP_NUM_THREADS to 4
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"]="8"

In [80]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import scipy
import cftime
import cartopy
import cartopy.crs as ccrs
import matplotlib.colors as colors
import cmaps
from xhistogram.xarray import histogram
from statsmodels.nonparametric.smoothers_lowess import lowess as  sm_lowess
from copy import copy
import geopandas as gpd
from shapely.geometry import Point

In [81]:
import datetime

In [82]:
import rpy2
from rpy2.robjects.packages import importr, data
from rpy2.robjects import pandas2ri
import rpy2.robjects as ro
from rpy2.robjects import globalenv


%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [83]:
%R Sys.setenv(OMP_NUM_THREADS = 8)
%R Sys.setenv(OPENBLAS_NUM_THREADS = 8)
%R Sys.setenv(MKL_NUM_THREADS = 8)

1


In [84]:
extRemes = importr("extRemes")
twosamples = importr("twosamples")
dplyr = importr("dplyr")
%R source("R_utils.R")

In [85]:
state_borders =\
cartopy.feature.NaturalEarthFeature(category='cultural',\
        name='admin_1_states_provinces_lakes', scale='50m', facecolor='none')

In [86]:
import warnings
warnings.filterwarnings('ignore')
%R options( warn = -1 )

In [87]:
%%R -w 800 --type=cairo
 extremes_bootLS <- function(r_df,fevd_options,n,alpha,rp){

  ntime <- length(r_df[['TXx']])

  boot_RL <- array(NA, dim = c(ntime, length(rp),n))
  boot_Loc <- matrix(data = NA,nrow=ntime,ncol=n)
  boot_Scale <- matrix(data = NA,nrow=ntime,ncol=n)

  boot_LocDiff <- matrix(nrow=1,ncol=n)
  boot_ScaleDiff <- matrix(nrow=1,ncol=n)
  boot_RLDiff <- matrix(nrow=length(rp),ncol=n)
  RiskRatios <- matrix(nrow=length(rp),ncol=n)

  i<-1
  while (i < n){
      index <- 1:ntime
      bx <- sample(index, size = ntime, replace = TRUE)
      r_df['bglost']<- r_df[['glost']][bx]
      r_df['bTXx']<- r_df[['TXx']][bx]
      r_df['btime'] <- r_df[['time']][bx]
      initial_vals <- list(location = mean(r_df$bTXx), scale = sd(r_df$bTXx), shape = 0.1)
      call_fevd <- paste("fevd(bTXx, data = r_df,",gsub("glost", "bglost", fevd_options),")")
      fit <- eval(parse(text = call_fevd))            
      if (length(which(findpars(fit)$scale<=0))==0){
          boot_rl <- return.level(fit, return.period = rp)
          boot_Loc[,i] <- findpars(fit)$location[order(r_df$btime)]
          boot_Scale[,i] <- findpars(fit)$scale[order(r_df$btime)]
          boot_RL[,,i] <- boot_rl[order(r_df$btime),]
          boot_LocDiff[i] <- boot_Loc[ntime,i] - boot_Loc[1,i]
          boot_ScaleDiff[i] <- boot_Scale[ntime,i] - boot_Scale[1,i]
          boot_RLDiff[,i] <- boot_RL[ntime,,i] - boot_RL[1,,i]

          i <- i+1
      }     
      }     
  Diffloc_q <- apply(boot_LocDiff, 1, quantile, probs = c(alpha/2, 0.5, 1-alpha/2), na.rm = TRUE)
  Diffsca_q <- apply(boot_ScaleDiff, 1, quantile, probs = c(alpha/2, 0.5, 1-alpha/2), na.rm = TRUE)
  DiffRL_q <- t(apply(boot_RLDiff, 1, quantile, probs = c(alpha/2, 0.5, 1-alpha/2), na.rm = TRUE))
  loc_q <- t(apply(boot_Loc, 1, quantile, probs = c(alpha/2, 0.5, 1-alpha/2), na.rm = TRUE))
  sca_q <- t(apply(boot_Scale, 1, quantile, probs = c(alpha/2, 0.5, 1-alpha/2), na.rm = TRUE))
  RL_q <- apply(boot_RL, c(1,2), quantile, probs = c(alpha/2, 0.5, 1-alpha/2), na.rm = TRUE)

  return(list(Dloc_q=Diffloc_q,Dsca_q=Diffsca_q,DRL_q=DiffRL_q,loc_q=loc_q,sca_q=sca_q,RL_q=RL_q))
  }
##########################  

In [88]:
%%R -w 800 --type=cairo

extremes_bootGE_ <- function(m_gev,r_df,fevd_options,n,alpha,rp){
   ### Following E.Gilleland 2020.
   ntime <- length(r_df[['TXx']])
   fitter <- function(data,...,Fit){
      np <- length(rp)

      RL_q <- array(NA, dim = c(ntime, length(rp),3))
      DiffRL_q <- array(NA, dim = c(length(rp),3))

      data <- data.frame(TXx=data,Fit$cov.data)
      initial_vals <- list(location = mean(data$TXx), scale = sd(data$TXx), shape = 0.1)
      call_fevd <- paste("fevd(TXx, data = data,",fevd_options,")")
      fit <- eval(parse(text = call_fevd))
      if (length(which(findpars(fit)$scale<=0))==0){
        Loc <- findpars(fit)$location
        Scale <- findpars(fit)$scale
        DiffLoc <- findpars(fit)$location[ntime] - findpars(fit)$location[1]
        DiffScale <- findpars(fit)$scale[ntime] - findpars(fit)$scale[1]
        RL <- return.level(fit,return.period=rp)
        DiffRL <- RL[ntime,] - RL[1,]
        out <- c(DiffLoc,DiffScale,DiffRL,Loc,Scale,RL)
        return(out)
        }
    }
   simphx<-function(size,...,Fit){
            out<-c(rextRemes(Fit,size))
            return(out)
            }
   pboopted<-pbooter(x=r_df$TXx,statistic=fitter,B=n,rmodel=simphx,Fit=m_gev,verbose=FALSE)

   res<- ci(pboopted,na.rm=TRUE,type="perc",alpha=alpha)
   #
   Diffloc_q <- cbind(res$perc[1,1],res$perc[1,2],res$perc[1,3])
   Diffsca_q <- cbind(res$perc[2,1],res$perc[2,2],res$perc[2,3])
   DiffRL_q <- cbind(res$perc[3:as.integer(3+length(rp)-1),1],res$perc[3:integer(3+length(rp)-1),2],res$perc[3:integer(3+length(rp)-1),3])
   #              
   ib <- as.integer(3+length(rp))
   ie <- as.integer(3+length(rp)+ntime-1)
   loc_q <- cbind(res$perc[ib:ie,1],res$perc[ib:ie,2],res$perc[ib:ie,3])

   ib <- as.integer(3+length(rp)+ntime)
   ie <- as.integer(3+length(rp)+ntime*2-1)
   sca_q <- cbind(res$perc[ib:ie,1],res$perc[ib:ie,2],res$perc[ib:ie,3])

   ib <- as.integer(3+length(rp)+ntime*2)
   ie <- as.integer(dim(res$perc)[1])
   RL_q <- cbind(res$perc[ib:ie,1],res$perc[ib:ie,2],res$perc[ib:ie,3])
   #

   return(list(Dloc_q=Diffloc_q,Dsca_q=Diffsca_q,DRL_q=DiffRL_q,loc_q=loc_q,sca_q=sca_q,RL_q=RL_q))
  }

In [89]:
%%R -w 800 --type=cairo
#===============================================================================
#===============================================================================
#===============================================================================
# function to define indices of rows of dataframe that assign observations to splits
# This function was written by Finn Lindgren
# ARGS:
# N: positive integer; (= number of rows of full dataframe)
# K: positive integer; (= number of splits for cross-validation)
# Value/Output: numeric of length N with indices 1,...,K defining random splitting into K subsets
cv_indices_splits <- function(N, K=K){
  split_sizes <-  ceiling((1:K)*N/K) - ceiling((0:(K-1))*N/K)
  sample(rep(1:K, times=split_sizes), size=N, replace=FALSE)
}
#===============================================================================
# function that takes data frame and splits/partitions in to training and test dataframes
# This function was written by Finn Lindgren
# ARGS:
# data: a data frame
# splits: index vector defining the K data splits (as produced by cv_indices_splits)
# k: the index k \in 1:K for the subset to be used as test/validation set
# Value/Output:
# a list with two named elements;
# 1.) Train: training data frame with rows of data in subsets with split index not equal to k
# 2.) Test: test data frame with rows of data in subsets with split index equal to k
cv_split <- function(data, splits, k){
  list(Train =  data[((splits != k)&((splits != 0))),  ,drop=FALSE],
       Test  =  data[splits == k,  ,drop=FALSE][1:length(which(splits==k))-1,])

}
cv_split_modulo <- function(data, K, ik){
  list(Train =  data[c(as.integer(K*(ik-1)+1):as.integer(K*(ik-1)+floor(K/2)),as.integer(K*(ik-1)+1+floor(K/2+1)):as.integer(K*ik)),] ,
       Test  =  data[as.integer(K*(ik-1)+floor(K/2+1)),])
}

In [90]:
%%R -w 800 --type=cairo

cross_validation <- function(data,fevd_options,K){

#===============================================================================
# function that calls Mod_Train_Test for all for each of the K splits and collects the results
# ARGS:
# data: full data frame to be split in in to training/test sets
# splits: the splitting indices (as produced by cv_indices_splits)
# formula_list: a list of formulas as required by evgam

  AIC_cv <- NULL
  BIC_cv <- NULL
  scores <- list()  # note AIC/BIC need dealt with
  tests <- list()
  for(k in 1:K){
    data_k <- cv_split(data=data, splits=splits, k)
    initial_vals <- list(location = mean(data_k$Train[['TXx']]), scale = sd(data_k$Train[['TXx']]), shape = 0.1)
    call_fevd <- paste("fevd(TXx, data = data_k$Train,",fevd_options,")")
    mod <- eval(parse(text = call_fevd))
    pp <- summary(mod, silent = TRUE)
    if (length(pp$par)==3){
        location <- rep(pp$par[['location']],times =length(data_k$Test$time))
        scale <- rep(pp$par[['scale']],times =length(data_k$Test$time))
        shape <- rep(pp$par[['shape']],times =length(data_k$Test$time))
    } else if ((length(pp$par)==4)){
        location <- pp$par[['mu0']]+pp$par[['mu1']]*data_k$Test$glost
        scale <- rep(pp$par[['scale']],times =length(data_k$Test$time))
        shape <- rep(pp$par[['shape']],times =length(data_k$Test$time))
    } else if ((length(pp$par)==5)){
        location <- pp$par[['mu0']]+pp$par[['mu1']]*data_k$Test$glost
        scale <- pp$par[['sigma0']]+pp$par[['sigma1']]*data_k$Test$glost
        shape <- rep(pp$par[['shape']],times =length(data_k$Test$time))
    }
    params <-list(location=location,scale=scale,shape=shape)
    params['cell'] <- 1
    TrainAndTest <- list(mod.scores=GEV_scores(data_k$Test,params), mod.AIC=pp$AIC, mod.BIC=pp$BIC)

    scores[[k]] <- cbind(TrainAndTest[[1]], time=data_k$Test$time,split_k = k)
    AIC_cv[k] <- TrainAndTest[[2]]
    BIC_cv[k] <- TrainAndTest[[3]]
    tests[[k]] <- cbind(KS_CVM(data_k$Test,params), time=data_k$Test$time, split_k = k)
  }
  scores <- do.call(rbind, scores)
  tests <- do.call(rbind, tests)
  list(scores_cv=scores, tests_cv=tests, AIC_cv=AIC_cv, BIC_cv=BIC_cv)
}

# READ TXx, DJF Temperature and topography FILES

In [91]:
yearb=1979
yeare=2025

lon_min = 190
lon_max = 220
lat_min = 53
lat_max = 72

ifile='~/data/TXx/AnnualMaximumDailyTmax.era5.1941.2025.nc4'

dso = xr.open_dataset(ifile)
dso = dso.drop_vars('TXx_day')
dso['time'] = np.arange(1941,2025+1,1)

dso = dso.sel(time=slice(f'{yearb-1}-01-01',f'{yeare}-02-01'))
dso = dso.sortby('lat')

units = 'K'

# K to Farenheit
dso['TXx'] = (dso['TXx'] - 273.15) * 9/5 + 32 
units = f'$\degree$ F'

In [ ]:


# Load the ERA5 land-sea mask and standardize its coordinate names.
lsmaskfile='~/data/ERA5/era5_lsmask.nc'
lsmask = xr.open_dataset(lsmaskfile)
lsm = lsmask['LSM'].isel(time=0).drop_vars('time')
lsm = lsm.rename({'longitude':'lon','latitude':'lat'})  
lsm = lsm.sortby('lat')

dso['TXx'] = dso['TXx'].where(lsm >= 0.5)

In [93]:
dso['TXx'] = dso['TXx'].where(lsm >= 0.5)

In [94]:
fileGLOST = '../../data/AFI/annual_LOESS_NOAA_GLOST_anomaly.nc'
ds_glost=xr.open_dataset(fileGLOST)
ds_glost = ds_glost.sel(time=slice(f'{yearb}',f'{yeare}'))
ann_glost = ds_glost['glost']
dso=dso.merge({'glost':ann_glost})

In [95]:
ds = dso.sel(lon=slice(lon_min,lon_max),lat=slice(lat_min,lat_max))
Tlon = ds.lon.data
Tlat = ds.lat.data

In [96]:
return_periods = [2, 5, 10, 20, 50, 100]
%R rp <- c(2, 5, 10, 20, 50, 100)
Tquantile = [0.05,0.5,0.95]
K = 5 # number of splits for Cross-Validation
%R -i K 


In [97]:
%R set.seed(1)

In [98]:
# Define arrays GEV output
location = xr.zeros_like(ds['TXx'])*np.nan
shape = xr.zeros_like(ds['TXx'])*np.nan
scale = xr.zeros_like(ds['TXx'])*np.nan
rl = xr.zeros_like(ds['TXx']).expand_dims(dim={'return_periods':return_periods})*np.nan
AIC = xr.zeros_like(ds['TXx'].mean('time'))*np.nan
BIC = xr.zeros_like(ds['TXx'].mean('time'))*np.nan
pvalueL = xr.zeros_like(ds['TXx'].mean('time'))*np.nan
pvalueS = xr.zeros_like(ds['TXx'].mean('time'))*np.nan

# Define arrays Confidence Interval 
location_q = xr.zeros_like(ds['TXx']).expand_dims(quantile=Tquantile)*np.nan
scale_q = xr.zeros_like(ds['TXx']).expand_dims(quantile=Tquantile)*np.nan
rl_q = xr.zeros_like(ds['TXx']).expand_dims(quantile=Tquantile).expand_dims(dim={'return_periods':return_periods})*np.nan
Dlocation_q = xr.zeros_like(ds['TXx'].mean('time')).expand_dims(quantile=Tquantile)*np.nan
Dscale_q = xr.zeros_like(ds['TXx'].mean('time')).expand_dims(quantile=Tquantile)*np.nan
Drl_q = xr.zeros_like(ds['TXx'].mean('time')).expand_dims(quantile=Tquantile).expand_dims(dim={'return_periods':return_periods})*np.nan

# Define arrays Cross-Validation
SE_cv = xr.zeros_like(ds['TXx'])*np.nan
DawSeb_cv = xr.zeros_like(ds['TXx'])*np.nan
CRPS_cv = xr.zeros_like(ds['TXx'])*np.nan
WCRPS_cv = xr.zeros_like(ds['TXx'])*np.nan
KS_D_cv = xr.zeros_like(ds['TXx'])*np.nan
KS_pvalue_cv = xr.zeros_like(ds['TXx'])*np.nan
CVM_D_cv = xr.zeros_like(ds['TXx'])*np.nan
CVM_pvalue_cv = xr.zeros_like(ds['TXx'].mean('time')).expand_dims(split=K)*np.nan
AIC_cv = xr.zeros_like(ds['TXx'].mean('time')).expand_dims(split=K)*np.nan
BIC_cv = xr.zeros_like(ds['TXx'].mean('time')).expand_dims(split=K)*np.nan

In [99]:
%R fevd_options <- "type='GEV', method='MLE', initial = initial_vals, location.fun = ~ glost, scale.fun = ~ glost"

array(["type='GEV', method='MLE', initial = initial_vals, location.fun = ~ glost, scale.fun = ~ glost"],
      dtype='<U93')

In [100]:
compute_CI = False

In [101]:
print(datetime.datetime.now())

2026-08-19 09:39:19.715258


In [102]:


mask=xr.where(np.isnan(ds['TXx'].mean('time'))==False,1,0)

idlat,idlon=np.where(mask==1)
for ij in range(0,len(idlon)):
  jj=idlat[ij]
  ii=idlon[ij]
  ilon=Tlon[ii]
  ilat=Tlat[jj]
  # Extract values for boxes
  dsij=ds.sel(lon=ilon,lat=ilat)
  df = dsij.to_dataframe()
  df = df.reset_index()

  df= df.assign(cell=1)
  # Remove zeros and NAN 
  # Remove timeseries with less than 15 data (as in Bilotta et al.)
  # Save rows removed
  dfsave = df[(df['TXx'].isna())]
  df = df[df['TXx'].notna()]

  itime = df['time'].values
  ntime = len(itime)
  %R -i ntime
  if df.empty == False: 
    # Convert xarray to pandas DataFrame to R data.frame
    # You can now work with 'r_df' in R

    with (ro.default_converter + pandas2ri.converter).context():
      r_df = ro.conversion.get_conversion().py2rpy(df)
    globalenv['r_df'] = r_df

    # Extremes GEV model fitting  
    %R initial_vals <- list(location = mean(r_df[['TXx']]), scale = sd(r_df[['TXx']]), shape = 0.1)
    %R call_fevd <- paste("fevd(TXx, data = r_df,",fevd_options,")")
    %R m_gev <- eval(parse(text = call_fevd))

    %R m_gevL <- fevd(TXx, data = r_df,type='GEV',method='MLE', initial = initial_vals, location.fun= ~ glost)
    %R m_gevS <- fevd(TXx, data = r_df,type='GEV',method='MLE', initial = initial_vals )

    %R pp <- summary(m_gev, silent = TRUE)
    %R rpvalueL <- lr.test(m_gevS, m_gevL)$p.value
    %R rpvalueS <- lr.test(m_gevL, m_gev)$p.value
    %R -o rpvalueL,rpvalueS

    %R loc <- findpars(m_gev)$location
    %R sca <- findpars(m_gev)$scale
    %R sha <- findpars(m_gev)$shape
    %R rAIC <- pp$AIC
    %R rBIC <- pp$BIC
    %R -o loc,sca,sha,rAIC,rBIC

    if sca.all() > 0.:

        %R gev_rl <- return.level(m_gev, return.period = rp)
        %R -o gev_rl
        
        ######
        # Cross-Validation
        ######
        %R splits <- rep(c(1,2,3,4,5),times =ceiling(ntime/5))[1:ntime]
        #%R splits[as.integer(ntime-5):ntime] <- 0
        %R tests_scores <- cross_validation(r_df,fevd_options,K)
        %R ktime <- tests_scores$scores_cv$time
        %R df_ks_cvm <- tests_scores$tests
        %R SE <- tests_scores$scores_cv$SE
        %R DawSeb <- tests_scores$scores_cv$DawSeb
        %R CRPS <- tests_scores$scores_cv$CRPS      
        %R rAIC_cv <- tests_scores$AIC_cv
        %R rBIC_cv <- tests_scores$BIC_cv
        %R -o ktime,SE,DawSeb,CRPS,df_ks_cvm,rAIC_cv,rBIC_cv

        ######
        # Confidence Intervals
        ######
        if compute_CI == True:
          %R boot <- extremes_bootLS(r_df,fevd_options,300,0.1,rp)
          %R DLoc_qr <- boot$Dloc_q
          %R DScale_qr <- boot$Dsca_q
          %R DRL_qr <- boot$DRL_q
          %R Loc_qr <- boot$loc_q
          %R Scale_qr <- boot$sca_q
          %R RL_qr <- boot$RL_q
          %R -o DLoc_qr,DScale_qr,DRL_qr,Loc_qr,Scale_qr,RL_qr


        ######
        # Write output
        ######

        location.loc[{'time':itime,'lat':ilat,'lon':ilon}] = loc
        scale.loc[{'time':itime,'lat':ilat,'lon':ilon}] = sca
        shape.loc[{'time':itime,'lat':ilat,'lon':ilon}] = sha
        for irp in range(0,len(return_periods)):
          rl.loc[{'time':itime,'lat':ilat,'lon':ilon,'return_periods':return_periods[irp]}] = gev_rl[:,irp]
        AIC.loc[{'lat':ilat,'lon':ilon}] = rAIC.item()
        BIC.loc[{'lat':ilat,'lon':ilon}] = rBIC.item()
        pvalueL.loc[{'lat':ilat,'lon':ilon}] = rpvalueL.item()
        pvalueS.loc[{'lat':ilat,'lon':ilon}] = rpvalueS.item()

        # CV
        SE_cv.loc[{'time':ktime,'lat':ilat,'lon':ilon}] = SE
        DawSeb_cv.loc[{'time':ktime,'lat':ilat,'lon':ilon}] = DawSeb
        CRPS_cv.loc[{'time':ktime,'lat':ilat,'lon':ilon}] = CRPS
        KS_D_cv.loc[{'time':ktime,'lat':ilat,'lon':ilon}] = df_ks_cvm['KS_D'].values
        KS_pvalue_cv.loc[{'time':ktime,'lat':ilat,'lon':ilon}] = df_ks_cvm['KS_pvalue'].values
        CVM_D_cv.loc[{'time':ktime,'lat':ilat,'lon':ilon}] = df_ks_cvm['CVM_D'].values
        CVM_pvalue_cv.loc[{'time':ktime,'lat':ilat,'lon':ilon}] = df_ks_cvm['CVM_pvalue'].values
        AIC_cv.loc[{'lat':ilat,'lon':ilon}] = rAIC_cv
        BIC_cv.loc[{'lat':ilat,'lon':ilon}] = rBIC_cv

        # CI
        if compute_CI == True:
          Dlocation_q.loc[{'lat':ilat,'lon':ilon}] = DLoc_qr[0,:].T
          Dscale_q.loc[{'lat':ilat,'lon':ilon}] = DScale_qr[0,:].T
          location_q.loc[{'time':itime,'lat':ilat,'lon':ilon}] = Loc_qr.T
          scale_q.loc[{'time':itime,'lat':ilat,'lon':ilon}] = Scale_qr.T
          for irp in range(0,len(return_periods)):
            Drl_q.loc[{'lat':ilat,'lon':ilon,'return_periods':return_periods[irp]}] = DRL_qr[irp,:]
            rl_q.loc[{'time':itime,'lat':ilat,'lon':ilon,'return_periods':return_periods[irp]}] = RL_qr[:,:,irp]

KeyError: "not all values found in index 'time'. Try setting the `method` keyword argument (example: method='nearest')."

In [106]:
%R print(df)

function (x, df1, df2, ncp, log = FALSE) 
{
    if (missing(ncp)) 
        .Call(C_df, x, df1, df2, log)
    else .Call(C_dnf, x, df1, df2, ncp, log)
}
<environment: namespace:stats>


<rpy2.robjects.functions.SignatureTranslatedFunction object at 0x3d3b91860> [3]
R classes: ('function',)

In [ ]:
print(datetime.datetime.now())

In [ ]:
dsout=rl.to_dataset(name='rl')
dsout=dsout.merge({'location':location,'scale':scale,'shape':shape})
dsout=dsout.merge({'AIC':AIC,'BIC':BIC,'pvalueL':pvalueL,'pvalueS':pvalueS})
#
if compute_CI == True:
  dsout=dsout.merge({'location_q':location_q})
  dsout=dsout.merge({'scale_q':scale_q})
  dsout=dsout.merge({'rl_q':rl_q})
  dsout=dsout.merge({'Difflocation_q':Dlocation_q})
  dsout=dsout.merge({'Diffscale_q':Dscale_q})
  dsout=dsout.merge({'Diffrl_q':Drl_q})
#
dsout=dsout.merge({'CRPS_cv':CRPS_cv})
dsout=dsout.merge({'DawSeb_cv':DawSeb_cv,'SE_cv':SE_cv})
dsout=dsout.merge({'KS_D_cv':KS_D_cv,'KS_pvalue_cv':KS_pvalue_cv,'CVM_D_cv':CVM_D_cv,'CVM_pvalue_cv':CVM_pvalue_cv})
dsout=dsout.merge({'AIC_cv':AIC_cv,'BIC_cv':BIC_cv})
#
dsout.to_netcdf(f'~/data/TXx/{itest}_{yearb}_{yeare}.nc')